# DPE Energy Label — starter

**Is this dwelling's energy label E, F or G?** 100,000 real energy
performance diagnoses (DPE) published by ADEME, described the way the
diagnostician recorded them.

In this challenge **the model is fixed**. The scorer always fits
scikit-learn's default `LogisticRegression()` on the features you submit,
and ranks the test dwellings by ROC AUC. You do not submit predictions;
you submit a **feature matrix**. Every point of AUC comes from
preprocessing.

This notebook does the plumbing once, end to end, with the most naive
features possible:

1. get the data
2. read it
3. build a (naive) feature matrix for train **and** test rows
4. check it locally with the scorer's own model
5. write `submission.csv.gz` and submit

Everything after that is yours. Read `EXPERTISE.md` before you start
cleaning: it explains what the regulation says about each column.

---

## 0. Setup

The ML-Arena client is published as **`mlarena-sdk`** and imports as
`mlarena`. Do not `pip install mlarena`: that is an unrelated package.

The scorer runs **scikit-learn 1.8.0**. Install the same version if you
want your local numbers to match the leaderboard.

In [ ]:
!pip install -q mlarena-sdk scikit-learn==1.8.0

---

## 1. Get the data

Paste your personal API key from your ML-Arena **Profile** page (it
starts with `mlk_user_`) and the challenge id, the number at the end of
the challenge page's address. `download_dataset` writes five files into
the working directory: `train.csv.gz`, `test.csv.gz`,
`sample_submission.csv.gz`, `EXPERTISE.md` and `DICTIONNAIRE.md`.

In [ ]:
import mlarena

API_KEY = "mlk_user_..."   # <- paste yours here
CHALLENGE_ID = 191         # DPE Energy Label: https://ml-arena.com/viewchallenge/191

assert CHALLENGE_ID is not None, "set CHALLENGE_ID to the number in the challenge page URL"
client = mlarena.connect(api_key=API_KEY)
client.download_dataset(CHALLENGE_ID, ".")

---

## 2. Read it

`train.csv.gz` has the target `classe_efg` (1 = label E, F or G);
`test.csv.gz` has the same columns without it. pandas reads the gzip
directly.

In [ ]:
import numpy as np
import pandas as pd

train = pd.read_csv("train.csv.gz", low_memory=False)
test = pd.read_csv("test.csv.gz", low_memory=False)
y = train["classe_efg"]

print("train", train.shape, " test", test.shape)
print(f"E/F/G share in train: {y.mean():.3f}")
train.head()

About 100 columns, most of them text, many of them mostly empty. Look at
what pandas made of them before deciding anything. `is_numeric_dtype`
is the reliable test: with pandas 3, text columns are not `object`.

In [ ]:
from pandas.api.types import is_numeric_dtype

features = [c for c in train.columns if c not in ("id", "classe_efg")]
numeric = [c for c in features if is_numeric_dtype(train[c])]
text = [c for c in features if c not in numeric]
print(len(numeric), "numeric columns,", len(text), "text columns")

summary = pd.DataFrame({
    "dtype": train[features].dtypes.astype(str),
    "filled": train[features].notna().mean().round(3),
    "distinct": train[features].nunique(),
})
summary.sort_values("filled").head(15)

---

## 3. A naive feature matrix

The laziest thing that runs: keep the numeric columns exactly as pandas
read them, and replace every empty cell with 0. No cleaning, no scaling,
no text. This is the bottom rung, not a recommendation: the challenge
page lists what the documented steps are worth.

Whatever you build, build it with **one function applied to train and
test alike**, and learn anything data-dependent (a median, a vocabulary,
a scale) from the train rows only.

In [ ]:
def build_features(df):
    return df[numeric].fillna(0)

X_train = build_features(train)
X_test = build_features(test)
print("features:", X_train.shape[1])

---

## 4. Check it locally

The scorer does exactly this: `LogisticRegression()` with its defaults,
fitted on your train rows, ROC AUC on the test rows. You do not have the
test labels, so hold out part of train. (The official split keeps each
building, with all its flats, on one side; a random holdout does not, so
expect it to be slightly optimistic.)

In [ ]:
import warnings
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

def local_score(X, y):
    X_fit, X_val, y_fit, y_val = train_test_split(
        X, y, test_size=0.3, random_state=0, stratify=y)
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always")
        model = LogisticRegression().fit(X_fit, y_fit)   # the scorer's model, untouched
    converged = not any(issubclass(w.category, ConvergenceWarning) for w in caught)
    auc = roc_auc_score(y_val, model.predict_proba(X_val)[:, 1])
    print(f"holdout ROC AUC {auc:.4f}   converged: {converged}")
    return auc

local_score(X_train, y)

If `converged` is `False`, the solver stopped at its 100-iteration limit
before reaching the optimum. The leaderboard will warn you too. With
columns on wildly different scales, that is what happens.

---

## 5. Write the submission

One file, **`submission.csv.gz`**, gzip-compressed:

- column `id`, then 1 to 300 numeric feature columns with no NaN or inf;
- **every** test id;
- train ids: all of them, or any subset of **at least 20,000** (dropping
  rows you do not trust is allowed);
- the scorer uses its own copy of the labels, so do not include
  `classe_efg`.

`to_csv` compresses from the `.gz` extension. `index=False` matters: an
unnamed index column is rejected.

In [ ]:
submission = pd.concat([
    X_train.assign(id=train["id"]),
    X_test.assign(id=test["id"]),
], ignore_index=True)
submission = submission[["id"] + list(X_train.columns)]

submission.to_csv("submission.csv.gz", index=False)
print(submission.shape)

Check it before uploading. Each of these is a rule the scorer enforces;
a rejected file costs you one of your daily submissions.

In [ ]:
values = submission.drop(columns="id")
assert submission["id"].is_unique, "duplicate ids"
assert set(test["id"]) <= set(submission["id"]), "a test id is missing"
assert submission["id"].str.startswith("tr_").sum() >= 20_000, "fewer than 20,000 train rows"
assert 1 <= values.shape[1] <= 300, "1 to 300 feature columns"
assert all(is_numeric_dtype(values[c]) for c in values), "a text column"
assert np.isfinite(values.to_numpy(dtype=float)).all(), "NaN or inf"
print("submission.csv.gz looks well-formed")

---

## 6. Submit

The file you upload must be named exactly `submission.csv.gz`.
`submit` returns as soon as the file is deployed; the score appears on
the leaderboard a minute or two later.

In [ ]:
result = client.submit(challenge_id=CHALLENGE_ID, files=["submission.csv.gz"])
print(result)

In [ ]:
client.leaderboard(CHALLENGE_ID).head(10)

---

## 7. Where to go from here

The model will never change, so the question for every column is: *what
numbers would let one straight line use this?* `EXPERTISE.md` gives the
regulatory reasons behind the choices that matter; `DICTIONNAIRE.md`
tells you what each column is, how often it is filled, and what its
values look like.

Measure one change at a time with `local_score`, and keep the ones that
move it.

**Rules.** Features must be computed from the columns you were given.
Target statistics are allowed only out-of-fold within train. No external
data and no labels from ADEME's public base, and no model other than the
scorer's smuggled in as a feature. Your notebook is part of the grade.